In [0]:
# ==========================================
# 03_gold_aggregate — Silver -> Gold
# ==========================================

# ----------------------------
# Standard widgets (all layers)
# ----------------------------
dbutils.widgets.text("base_path", "/Volumes/workspace/ecommerce/ecommerce_data")
dbutils.widgets.text("raw_path", "/Volumes/workspace/ecommerce/ecommerce_data/2019-Nov.csv")  # not used here
dbutils.widgets.text("bronze_path", "")  # not used here
dbutils.widgets.text("silver_path", "")
dbutils.widgets.text("gold_path", "")
dbutils.widgets.dropdown("mode", "incremental", ["incremental", "full"])

base_path = dbutils.widgets.get("base_path").strip()
silver_path = dbutils.widgets.get("silver_path").strip()
gold_path = dbutils.widgets.get("gold_path").strip()
mode = dbutils.widgets.get("mode").strip()
#mode = "full"
if not silver_path:
    silver_path = f"{base_path}/silver/events"
if not gold_path:
    gold_path = f"{base_path}/gold/product_performance_daily"

print(f"silver_path={silver_path}")
print(f"gold_path={gold_path}")
print(f"mode={mode}")

from pyspark.sql import functions as F
from delta.tables import DeltaTable


# ----------------------------
# Helpers
# ----------------------------
def delta_path_exists(path: str) -> bool:
    try:
        dbutils.fs.ls(path)
        return True
    except Exception:
        return False

def require(value, name):
    if value is None or str(value).strip() == "":
        raise ValueError(f"Missing required parameter: {name}")
    return value

require(silver_path, "silver_path")
require(gold_path, "gold_path")


# ----------------------------
# Determine impacted dates
# ----------------------------
# Prefer dates passed from Silver task (multi-task workflow)
impacted_dates_str = ""
try:
    # Task key must match your workflow task name. If you named it 'silver_layer', use that.
    impacted_dates_str = dbutils.jobs.taskValues.get(taskKey="silver_layer", key="impacted_dates", default="")
except Exception:
    impacted_dates_str = ""

impacted_dates = [d for d in impacted_dates_str.split(",") if d.strip()]

silver_df = spark.read.format("delta").load(silver_path)

if mode == "incremental" and impacted_dates:
    print(f"Incremental: recomputing Gold only for impacted dates = {impacted_dates[:10]}{'...' if len(impacted_dates) > 10 else ''}")
    silver_df = silver_df.filter(F.col("event_date").isin(impacted_dates))
elif mode == "incremental":
    # Fallback if taskValues not available (e.g., running notebook manually)
    # Recompute last 1 day of data present in Silver (conservative)
    print("Incremental: no impacted_dates received; falling back to last 1 day window.")
    max_date = silver_df.select(F.max("event_date").alias("mx")).collect()[0]["mx"]
    if max_date is not None:
        silver_df = silver_df.filter(F.col("event_date") == F.lit(max_date))
    else:
        print("Silver appears empty; exiting.")
        dbutils.notebook.exit("NO_SILVER_DATA")
else:
    print("Full mode: recomputing Gold for entire Silver dataset.")


# ----------------------------
# Aggregate (Gold logic)
# ----------------------------
# Output grain: event_date x product_id (plus optional dims)
gold_daily = (
    silver_df
    .groupBy("event_date", "product_id", "category_id", "category_code", "brand")
    .agg(
        F.sum(F.when(F.col("event_type") == "view", 1).otherwise(0)).cast("long").alias("views"),
        F.sum(F.when(F.col("event_type") == "cart", 1).otherwise(0)).cast("long").alias("carts"),
        F.sum(F.when(F.col("event_type") == "purchase", 1).otherwise(0)).cast("long").alias("purchases"),
        F.sum(F.when(F.col("event_type") == "purchase", F.col("price")).otherwise(F.lit(0.0))).alias("revenue"),
        F.countDistinct("user_id").cast("long").alias("unique_users"),
        F.countDistinct("user_session").cast("long").alias("unique_sessions"),
    )
    .withColumn("avg_purchase_value",
                F.when(F.col("purchases") > 0, F.col("revenue") / F.col("purchases")).otherwise(F.lit(None)))
    .withColumn("processing_ts", F.current_timestamp())
)

gold_count = gold_daily.count()
print(f"Gold rows to write = {gold_count}")

if gold_count == 0:
    print("No gold rows produced; exiting cleanly.")
    dbutils.notebook.exit("NO_GOLD_DATA")


# ----------------------------
# Write / Merge into Gold
# ----------------------------
# Partition by event_date for efficient date-based refresh
if mode == "full":
#if mode == "full" or not delta_path_exists(gold_path):
    (
        gold_daily.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .partitionBy("event_date")
        .save(gold_path)
    )
    print("Gold written in overwrite mode (full refresh / first run).")
    dbutils.notebook.exit("Full_refresh_done")
else:
    # MERGE by natural key: event_date + product_id (+ optional dims)
    # Because we partition by event_date, this supports efficient re-writes for impacted dates.
    delta_gold = DeltaTable.forPath(spark, gold_path)

    merge_condition = """
      t.event_date = s.event_date
      AND t.product_id = s.product_id
      AND t.category_id <=> s.category_id
      AND t.category_code <=> s.category_code
      AND t.brand <=> s.brand
    """

    (
        delta_gold.alias("t")
        .merge(gold_daily.alias("s"), merge_condition)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )
    print("Gold MERGE completed (incremental).")


dbutils.jobs.taskValues.set(key="gold_rowcount", value=int(gold_count))
print("Gold job completed.")


In [0]:
display(
  spark.sql("""
    SELECT product_id, SUM(revenue) AS total_revenue
    FROM delta.`/Volumes/workspace/ecommerce/ecommerce_data/gold/product_performance_daily`
    GROUP BY product_id
    ORDER BY total_revenue DESC
    LIMIT 10
  """)
)